# Pydantic AI basics: seeing every step of the agent loop

This notebook exists to understand **how the harness works** before wiring it into the AERONET agent.
Everything here uses three toy tools and the same GPT-5.6 Luna model as the AERONET notebook, so the
mechanics transfer one to one. There is no Docker sandbox involved.

What you will see, section by section:

1. **The loop and the message model.** One `run_sync` call, then a walk through everything it produced:
   `ModelRequest` / `ModelResponse` objects and their parts (user prompt, thinking, text, tool call, tool return).
2. **Token usage per model call**, including cached and reasoning tokens, and how per-call usage sums to the run total.
3. **Reasoning traces.** What the model exposes (a summary, never raw chain-of-thought) and the one setting that turns it on.
4. **Step-by-step execution with `agent.iter()`.** Each model call and each tool batch is printed *as it completes*,
   instead of receiving a finished result object at the end.


Run the cells top to bottom. Each section is self-contained after the setup cells.

In [55]:
import nest_asyncio
nest_asyncio.apply()          # lets agent.run_sync() work inside Jupyter's running event loop

## 0. Setup: model, tools, agent

Same model construction as the AERONET notebook.

> **Verified 2026-09-26:** Bedrock Mantle rejects `reasoning.summary` for `us.openai.gpt-5.6-luna`
> (HTTP 400, `unsupported_parameter`). So reasoning text is **not available** on this endpoint. Reasoning
> still happens and is still counted (`reasoning_tokens`), and the encrypted reasoning is round-tripped
> automatically, but every `ThinkingPart` will have empty `content`. Section 3 shows what remains.

In [56]:
from aws_bedrock_token_generator import provide_token
from pydantic_ai import Agent, UsageLimits, ModelRetry
from pydantic_ai.models.bedrock_mantle import BedrockMantleResponsesModel, BedrockMantleProvider
from pydantic_ai.models.openai import OpenAIResponsesModelSettings

REGION           = "us-west-2"
BASE_URL         = f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1"
MODEL_ID         = "us.openai.gpt-5.6-luna"
REASONING_EFFORT = "low"        # none | low | medium | high | xhigh | max  (medium keeps the demos fast)

model = BedrockMantleResponsesModel(
    MODEL_ID,
    provider=BedrockMantleProvider(base_url=BASE_URL, api_key=provide_token(region=REGION)),
    settings=OpenAIResponsesModelSettings(
        max_tokens=4000,
        openai_reasoning_effort=REASONING_EFFORT,
        openai_reasoning_summary="auto",   # rejected by Bedrock Mantle for Luna (400 unsupported_parameter)
        openai_prompt_cache_key="basics:v1",
    ),
)
print(model.model_name, "via", model.client.base_url)

us.openai.gpt-5.6-luna via https://bedrock-runtime.us-west-2.amazonaws.com/openai/v1/


Three deliberately simple tools:

* `add` is synchronous. Pydantic AI runs sync tools in a worker thread, so they never block the event loop.
* `city_info` is **async and slow** (2 s sleep). Used in section 6 to prove that parallel tool calls really overlap.
* `divide` raises `ModelRetry` on division by zero. Used in section 7 to show what a retry looks like.

The docstring and the type hints become the JSON schema the model sees. Nothing else is needed.

In [57]:
import asyncio

CITIES = {
    "kathmandu": {"country": "Nepal",  "elevation_m": 1400, "population": 845_767},
    "nairobi":   {"country": "Kenya",  "elevation_m": 1795, "population": 4_397_073},
    "cairo":     {"country": "Egypt",  "elevation_m": 23,   "population": 9_539_673},
    "baltimore": {"country": "USA",    "elevation_m": 10,   "population": 585_708},
}

INSTRUCTIONS = (
    "You are a careful assistant. Always use the tools for arithmetic and for facts about cities; "
    "never compute or recall them yourself. Keep answers short."
)

# Preamble rule: the model writes one plain-text sentence BEFORE its tool calls, in the same response.
# One sentence covers all tools called at once. Pydantic AI keeps it as a TextPart in front of the ToolCallParts.
PREAMBLE_RULE = (
    " Whenever you call tools, first write ONE short sentence, as normal text before the tool calls, saying what "
    "you are about to do and why. If you call several tools at once, write a single sentence covering all of them. "
    "Never call a tool without that sentence."
)
INSTRUCTIONS = INSTRUCTIONS + PREAMBLE_RULE


In [58]:
agent = Agent(model, instructions=INSTRUCTIONS, retries=2)

In [59]:
@agent.tool_plain
def add(a: float, b: float) -> float:
    '''Add two numbers.'''
    return a + b

@agent.tool_plain
async def city_info(city: str) -> dict:
    '''Look up country, elevation (m) and population for a city. Slow: takes about two seconds.'''
    await asyncio.sleep(2)
    try:
        return CITIES[city.strip().lower()]
    except KeyError:
        raise ModelRetry(f"Unknown city {city!r}. Known cities: {', '.join(sorted(CITIES))}.")

@agent.tool_plain
def divide(numerator: float, denominator: float) -> float:
    '''Divide numerator by denominator.'''
    if denominator == 0:
        raise ModelRetry("Division by zero is undefined; ask the user for a non-zero denominator.")
    return numerator / denominator

print("tools:", [t for t in agent._function_toolset.tools])   # private attr, inspection only

tools: ['add', 'city_info', 'divide']


## 1. One run, then everything it produced

`run_sync` drives the whole loop to completion and hands back an `AgentRunResult`. The loop is:

```
UserPromptNode  ->  ModelRequestNode  ->  CallToolsNode  ->  ModelRequestNode  ->  ...  ->  End
  (your text)       (one API call)         (run tools)         (send results)
```

Every API call appends one `ModelResponse` to the history. Every tool batch appends one `ModelRequest`
containing the tool returns. The final answer is just the `TextPart` of the last `ModelResponse`.

In [60]:
r = agent.run_sync("What is 12.5 + 7.25?")
print("OUTPUT :", r.output)
print("USAGE  :", r.usage)          # RunUsage: totals over the whole run
print("RUN ID :", r.run_id)

OUTPUT : 19.75
USAGE  : RunUsage(cost=Decimal('0.00016588'), details={'reasoning_tokens': 0}, input_tokens=454, output_tokens=50, requests=2, tool_calls=1)
RUN ID : 01a0dc0f-eb20-71b9-acad-0666614f95fc


In [61]:
# The skeleton: which messages, and which parts inside each one
for i, m in enumerate(r.all_messages()):
    print(f"{i}: {type(m).__name__:14s} parts = {[type(p).__name__ for p in m.parts]}")

0: ModelRequest   parts = ['UserPromptPart']
1: ModelResponse  parts = ['TextPart', 'ToolCallPart']
2: ModelRequest   parts = ['ToolReturnPart']
3: ModelResponse  parts = ['TextPart']


### A printer that shows every part in full

This is the core of what you want in the AERONET notebook. It prints, for each message:

* `UserPromptPart` — what you sent
* `ThinkingPart` — reasoning summary text if the API returned one; otherwise only the encrypted signature exists
* `TextPart` — what the model wrote: the final answer, or, when the same response also has tool calls, the
  **preamble** sentence explaining the calls (printed as `PREAMBLE`)
* `ToolCallPart` — tool name, arguments, call id
* `ToolReturnPart` — the tool's result, its outcome flag and timestamp
* `RetryPromptPart` — the error text sent back when a tool raised `ModelRetry` or argument validation failed
* one usage line per `ModelResponse`: input / output / cache read / cache write / reasoning tokens / cost

In [62]:
from pydantic_ai.messages import (ModelRequest, ModelResponse, UserPromptPart, TextPart, ThinkingPart,
                                  ToolCallPart, ToolReturnPart, RetryPromptPart)

def usage_line(m: ModelResponse) -> str:
    u = m.usage
    reasoning = u.details.get("reasoning_tokens", 0)
    cost = f" ${u.cost:.5f}" if u.cost is not None else ""
    return (f"in {u.input_tokens} (cache read {u.cache_read_tokens}, cache write {u.cache_write_tokens}) "
            f"| out {u.output_tokens} (reasoning {reasoning}){cost}")

def show_part(p, clip=800, preamble=False):
    if isinstance(p, UserPromptPart):
        print(f"  >>> USER          : {p.content}")
    elif isinstance(p, ThinkingPart):
        txt = p.content.strip() if p.content else ""
        tag = "" if txt else "   <no summary text; encrypted signature only>"
        print(f"  ... THINKING      : {txt[:clip]}{tag}")
    elif isinstance(p, TextPart):
        label = "PREAMBLE      " if preamble else "MODEL TEXT    "
        print(f"  <<< {label}: {p.content[:clip]}")
    elif isinstance(p, ToolCallPart):
        print(f"  --> TOOL CALL     : {p.tool_name}({p.args})   id={p.tool_call_id}")
    elif isinstance(p, ToolReturnPart):
        print(f"  <-- TOOL RESULT   : {p.tool_name} [{p.outcome}] -> {str(p.content)[:clip]}")
    elif isinstance(p, RetryPromptPart):
        print(f"  !!! RETRY PROMPT  : {p.tool_name}: {str(p.content)[:clip]}")
    else:
        print(f"  ??? {type(p).__name__}: {p!r}"[:clip])

def show_messages(messages, clip=800):
    for i, m in enumerate(messages):
        if isinstance(m, ModelRequest):
            print(f"\n[{i}] ModelRequest  (sent to the model)")
        else:
            print(f"\n[{i}] ModelResponse (model={m.model_name}, finish={m.finish_reason}, id={m.provider_response_id})")
        preamble = isinstance(m, ModelResponse) and bool(m.tool_calls)   # text next to tool calls = preamble
        for p in m.parts:
            show_part(p, clip, preamble)
        if isinstance(m, ModelResponse):
            print(f"      usage: {usage_line(m)}")

In [63]:
show_messages(r.all_messages())


[0] ModelRequest  (sent to the model)
  >>> USER          : What is 12.5 + 7.25?

[1] ModelResponse (model=us.openai.gpt-5.6-luna, finish=stop, id=resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq)
  <<< PREAMBLE      : I’ll add the two numbers using the arithmetic tool.
  --> TOOL CALL     : add({"a":12.5,"b":7.25})   id=resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq:call_a4c491f5913d5bf6b8cd84738a8c70fa
      usage: in 199 (cache read 0, cache write 0) | out 43 (reasoning 0) $0.00010

[2] ModelRequest  (sent to the model)
  <-- TOOL RESULT   : add [success] -> 19.75

[3] ModelResponse (model=us.openai.gpt-5.6-luna, finish=stop, id=resp_iudvapfe64wt6o4ebdiflcytx5bj55zazu7xq6zle5fsjl3kglvq)
  <<< MODEL TEXT    : 19.75
      usage: in 255 (cache read 0, cache write 0) | out 7 (reasoning 0) $0.00007


In [64]:
r.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What is 12.5 + 7.25?', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 890629, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 891106, tzinfo=datetime.timezone.utc), instructions='You are a careful assistant. Always use the tools for arithmetic and for facts about cities; never compute or recall them yourself. Keep answers short. Whenever you call tools, first write ONE short sentence, as normal text before the tool calls, saying what you are about to do and why. If you call several tools at once, write a single sentence covering all of them. Never call a tool without that sentence.', run_id='01a0dc0f-eb20-71b9-acad-0666614f95fc', conversation_id='01a0dc0f-eb20-71b9-acad-06674667f03f'),
 ModelResponse(parts=[TextPart(content='I’ll add the two numbers using the arithmetic tool.', id='msg_5cabe4f19d6a5e889a3f6c1755da4fca', provider_name='bedrock-mantle', provider_details={'phase': 'commentary'}), ToolCa

### First Message

In [65]:
first_message = r.all_messages()[0]
first_message

ModelRequest(parts=[UserPromptPart(content='What is 12.5 + 7.25?', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 890629, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 891106, tzinfo=datetime.timezone.utc), instructions='You are a careful assistant. Always use the tools for arithmetic and for facts about cities; never compute or recall them yourself. Keep answers short. Whenever you call tools, first write ONE short sentence, as normal text before the tool calls, saying what you are about to do and why. If you call several tools at once, write a single sentence covering all of them. Never call a tool without that sentence.', run_id='01a0dc0f-eb20-71b9-acad-0666614f95fc', conversation_id='01a0dc0f-eb20-71b9-acad-06674667f03f')

In [66]:
print(f"First message: {first_message.parts[0].content}")

First message: What is 12.5 + 7.25?


### Second Message

In [67]:
r.all_messages()[1]

ModelResponse(parts=[TextPart(content='I’ll add the two numbers using the arithmetic tool.', id='msg_5cabe4f19d6a5e889a3f6c1755da4fca', provider_name='bedrock-mantle', provider_details={'phase': 'commentary'}), ToolCallPart(tool_name='add', args='{"a":12.5,"b":7.25}', tool_call_id='resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq:call_a4c491f5913d5bf6b8cd84738a8c70fa', id='fc_85bee55b1c245e8385622bf4d975f608', provider_name='bedrock-mantle')], usage=RequestUsage(cost=Decimal('0.00010054'), details={'reasoning_tokens': 0}, input_tokens=199, output_tokens=43), model_name='us.openai.gpt-5.6-luna', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 46, 623945, tzinfo=datetime.timezone.utc), provider_name='bedrock-mantle', provider_url='https://bedrock-runtime.us-west-2.amazonaws.com/openai/v1/', provider_details={'finish_reason': 'completed', 'timestamp': datetime.datetime(2026, 9, 26, 4, 53, 46, tzinfo=TzInfo(0)), 'service_tier': 'default'}, provider_response_id='resp_ddns27ro56h5wi

In [68]:
r.all_messages()[1].usage

RequestUsage(cost=Decimal('0.00010054'), details={'reasoning_tokens': 0}, input_tokens=199, output_tokens=43)

In [69]:
r.all_messages()[1].parts

[TextPart(content='I’ll add the two numbers using the arithmetic tool.', id='msg_5cabe4f19d6a5e889a3f6c1755da4fca', provider_name='bedrock-mantle', provider_details={'phase': 'commentary'}),
 ToolCallPart(tool_name='add', args='{"a":12.5,"b":7.25}', tool_call_id='resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq:call_a4c491f5913d5bf6b8cd84738a8c70fa', id='fc_85bee55b1c245e8385622bf4d975f608', provider_name='bedrock-mantle')]

In [76]:
r.all_messages()[1].parts[0]

TextPart(content='I’ll add the two numbers using the arithmetic tool.', id='msg_5cabe4f19d6a5e889a3f6c1755da4fca', provider_name='bedrock-mantle', provider_details={'phase': 'commentary'})

In [75]:
r.all_messages()[1].parts[1].tool_name, r.all_messages()[1].parts[1].args

('add', '{"a":12.5,"b":7.25}')

### Third Message

In [77]:
r.all_messages()[2]

ModelRequest(parts=[ToolReturnPart(tool_name='add', content=19.75, tool_call_id='resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq:call_a4c491f5913d5bf6b8cd84738a8c70fa', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 46, 625931, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 4, 53, 46, 626474, tzinfo=datetime.timezone.utc), instructions='You are a careful assistant. Always use the tools for arithmetic and for facts about cities; never compute or recall them yourself. Keep answers short. Whenever you call tools, first write ONE short sentence, as normal text before the tool calls, saying what you are about to do and why. If you call several tools at once, write a single sentence covering all of them. Never call a tool without that sentence.', run_id='01a0dc0f-eb20-71b9-acad-0666614f95fc', conversation_id='01a0dc0f-eb20-71b9-acad-06674667f03f')

In [78]:
r.all_messages()[2].parts

[ToolReturnPart(tool_name='add', content=19.75, tool_call_id='resp_ddns27ro56h5wipfduqlvk42ae5i2ob7xjlvtw27m6qnq2yncagq:call_a4c491f5913d5bf6b8cd84738a8c70fa', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 46, 625931, tzinfo=datetime.timezone.utc))]

In [79]:
r.all_messages()[2].parts[0].tool_name, r.all_messages()[2].parts[0].content

('add', 19.75)

# Final Message

In [80]:
r.all_messages()[3]

ModelResponse(parts=[TextPart(content='19.75', id='msg_50320a18787b5da18fbe0872f6e677ee', provider_name='bedrock-mantle', provider_details={'phase': 'final_answer'})], usage=RequestUsage(cost=Decimal('0.00006534'), details={'reasoning_tokens': 0}, input_tokens=255, output_tokens=7), model_name='us.openai.gpt-5.6-luna', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 47, 155447, tzinfo=datetime.timezone.utc), provider_name='bedrock-mantle', provider_url='https://bedrock-runtime.us-west-2.amazonaws.com/openai/v1/', provider_details={'finish_reason': 'completed', 'timestamp': datetime.datetime(2026, 9, 26, 4, 53, 46, tzinfo=TzInfo(0)), 'service_tier': 'default'}, provider_response_id='resp_iudvapfe64wt6o4ebdiflcytx5bj55zazu7xq6zle5fsjl3kglvq', finish_reason='stop', run_id='01a0dc0f-eb20-71b9-acad-0666614f95fc', conversation_id='01a0dc0f-eb20-71b9-acad-06674667f03f')

In [81]:
r.all_messages()[3].parts[0]

TextPart(content='19.75', id='msg_50320a18787b5da18fbe0872f6e677ee', provider_name='bedrock-mantle', provider_details={'phase': 'final_answer'})

In [82]:
r.all_messages()[3].parts[0].content

'19.75'

In [83]:
r.all_messages()[3].usage

RequestUsage(cost=Decimal('0.00006534'), details={'reasoning_tokens': 0}, input_tokens=255, output_tokens=7)

Notes on what you just saw:

* `ModelRequest[0]` carries your prompt. The `instructions` string is stored on the request too (`m.instructions`), not as a part.
* The first `ModelResponse` contains the tool call (and a `ThinkingPart` if reasoning was summarised).
* `ModelRequest[2]` is built by Pydantic AI from the tool's return value. You never construct it.
* The last `ModelResponse` has the `TextPart` that becomes `r.output`.
* `r.response` is a shortcut for that last `ModelResponse`.

## 2. Token usage per model call

Each `ModelResponse.usage` is a `RequestUsage` for that one API call. `r.usage` is a `RunUsage`
that also counts `requests` and `tool_calls`. Per-call values sum to the run total.

Field meanings (all tokens):

| field | meaning |
|---|---|
| `input_tokens` | full prompt size, **including** the cached part |
| `cache_read_tokens` | portion of the input served from the prompt cache |
| `cache_write_tokens` | portion written to the cache (OpenAI reports 0; caching is automatic) |
| `output_tokens` | everything generated, **including** reasoning |
| `details["reasoning_tokens"]` | the reasoning share of `output_tokens` |
| `cost` | USD estimate from genai-prices, `None` if the model is unknown to it |

In [84]:
r.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What is 12.5 + 7.25?', timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 890629, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 4, 53, 45, 891106, tzinfo=datetime.timezone.utc), instructions='You are a careful assistant. Always use the tools for arithmetic and for facts about cities; never compute or recall them yourself. Keep answers short. Whenever you call tools, first write ONE short sentence, as normal text before the tool calls, saying what you are about to do and why. If you call several tools at once, write a single sentence covering all of them. Never call a tool without that sentence.', run_id='01a0dc0f-eb20-71b9-acad-0666614f95fc', conversation_id='01a0dc0f-eb20-71b9-acad-06674667f03f'),
 ModelResponse(parts=[TextPart(content='I’ll add the two numbers using the arithmetic tool.', id='msg_5cabe4f19d6a5e889a3f6c1755da4fca', provider_name='bedrock-mantle', provider_details={'phase': 'commentary'}), ToolCa

In [85]:
from pydantic_ai.messages import ModelResponse

In [86]:
import pandas as pd

def usage_table(messages):
    rows = []
    for i, m in enumerate(messages):
        if isinstance(m, ModelResponse):
            u = m.usage
            rows.append({"msg": i, "model": m.model_name, "input": u.input_tokens, "cache_read": u.cache_read_tokens,
                         "cache_write": u.cache_write_tokens, "output": u.output_tokens,
                         "reasoning": u.details.get("reasoning_tokens", 0), "cost_usd": u.cost,
                         "tool_calls": len(m.tool_calls), "finish": m.finish_reason})
    return pd.DataFrame(rows)

df = usage_table(r.all_messages())
display(df)
print("column sums :", df[["input", "cache_read", "output", "reasoning"]].sum().to_dict())
print("r.usage     :", f"requests={r.usage.requests} tool_calls={r.usage.tool_calls} "
      f"input={r.usage.input_tokens} cache_read={r.usage.cache_read_tokens} output={r.usage.output_tokens} "
      f"details={r.usage.details}")

,msg,model,input,cache_read,cache_write,output,reasoning,cost_usd,tool_calls,finish
0,1,us.openai.gpt-5.6-luna,199,0,0,43,0,0.00010054,1,stop
1,3,us.openai.gpt-5.6-luna,255,0,0,7,0,0.00006534,0,stop


column sums : {'input': 454, 'cache_read': 0, 'output': 50, 'reasoning': 0}
r.usage     : requests=2 tool_calls=1 input=454 cache_read=0 output=50 details={'reasoning_tokens': 0}


## 4. Step by step with `agent.iter()`: print each step as it completes

`run_sync` hides the loop. `agent.iter()` exposes it as an async iterator over graph nodes:

* `UserPromptNode` — the run is starting
* `ModelRequestNode` — **about to call the model**; `node.request` is the `ModelRequest` being sent
  (your prompt on the first one, tool returns afterwards)
* `CallToolsNode` — **the model has answered**; `node.model_response` holds parts and usage, and the tools
  named in it are executed while this node runs
* `End` — the final output is in `node.data.output`, and `run.result` is the same `AgentRunResult` as before

So printing inside the loop gives you a live transcript with per-step usage, in the right order.
`run.usage` is available at every step (usage so far).

In [87]:
from pydantic_graph import End

async def ask_stepwise(question: str, history: list, max_requests: int = 10, clip: int = 800):
    '''Run one turn and print every step as it completes. Appends the turn to `history`. Returns the result.'''
    step = 0
    async with agent.iter(question, message_history=history,
                          usage_limits=UsageLimits(request_limit=max_requests)) as run:
        async for node in run:
            if Agent.is_user_prompt_node(node):
                print(f"=== TURN START: {question!r}")
            elif Agent.is_model_request_node(node):
                step += 1
                print(f"\n--- step {step}: sending to model ({len(node.request.parts)} part(s))")
                for p in node.request.parts:
                    show_part(p, clip)
            elif Agent.is_call_tools_node(node):
                m = node.model_response
                print(f"--- step {step}: model responded  ({len(m.tool_calls)} tool call(s))")
                for p in m.parts:
                    show_part(p, clip, preamble=bool(m.tool_calls))   # preamble sentence prints before its tool calls
                print(f"      usage this call : {usage_line(m)}")
                u = run.usage
                print(f"      usage so far    : requests={u.requests} tool_calls={u.tool_calls} "
                      f"in={u.input_tokens} out={u.output_tokens} cache_read={u.cache_read_tokens}")
            elif isinstance(node, End):
                print(f"\n=== FINAL ANSWER: {node.data.output}")
    result = run.result
    history.extend(result.new_messages())
    return result

history = []
r2 = await ask_stepwise("Look up Kathmandu and Nairobi and tell me the sum of their elevations.", history)

=== TURN START: 'Look up Kathmandu and Nairobi and tell me the sum of their elevations.'

--- step 1: sending to model (1 part(s))
  >>> USER          : Look up Kathmandu and Nairobi and tell me the sum of their elevations.
--- step 1: model responded  (2 tool call(s))
  ... THINKING      :    <no summary text; encrypted signature only>
  <<< PREAMBLE      : I’ll look up both cities’ elevations, then add them together.
  --> TOOL CALL     : city_info({"city":"Kathmandu"})   id=resp_3nhkpbkrgspzidbmpxpekya2ho5gy5zcpyplz5rk3whfoo7s7dvq:call_4012c68270e35071bd6ff42670c181b9
  --> TOOL CALL     : city_info({"city":"Nairobi"})   id=resp_3nhkpbkrgspzidbmpxpekya2ho5gy5zcpyplz5rk3whfoo7s7dvq:call_952cda558cd05b0ab935c30a1070d15b
      usage this call : in 201 (cache read 0, cache write 0) | out 93 (reasoning 20) $0.00017
      usage so far    : requests=1 tool_calls=0 in=201 out=93 cache_read=0

--- step 2: sending to model (2 part(s))
  <-- TOOL RESULT   : city_info [success] -> {'country': '

Things to notice:

* Tool **results** are printed when the *next* `ModelRequestNode` appears, because that is when they are packaged
  into a `ModelRequest`. If you want them the moment each tool finishes, use the event stream (next section).
* The `CallToolsNode` for the last step contains only text, no tool calls, so the loop ends.
* `run.result` is `None` until `End` is reached; after that it is a normal `AgentRunResult`.
* Top-level `await` works in Jupyter. Outside Jupyter wrap this in `asyncio.run(...)`.

### Preambles: a model-written sentence before every tool call

There is no reasoning text on Bedrock, so the substitute is a **preamble**: the instructions (cell above, `PREAMBLE_RULE`)
ask the model to write one short sentence before its tool calls, in the same response. Nothing is injected by code:

* The model emits `text` then `function_call` items. Pydantic AI stores them in order, as a `TextPart` followed by
  `ToolCallPart`s inside one `ModelResponse`.
* Because tool calls are present, the run continues (default `end_strategy="graceful"`); the text is **not** taken as
  the final answer. The last response has text and no tool calls, so that text becomes `result.output`.
* Several tools in one response get one sentence, because the rule says so.
* The instructions are sent with every request, so after each tool result the model writes a fresh preamble for the
  next calls, informed by what it just saw.

`show_part(..., preamble=True)` only changes the label. Re-run the three setup cells (`INSTRUCTIONS`, `Agent(...)`,
the tools) so the agent picks up the rule, then run the cell below.

In [88]:
history = []
r3 = await ask_stepwise("Look up Cairo and Baltimore, add their elevations, then divide the sum by 2.", history)

=== TURN START: 'Look up Cairo and Baltimore, add their elevations, then divide the sum by 2.'

--- step 1: sending to model (1 part(s))
  >>> USER          : Look up Cairo and Baltimore, add their elevations, then divide the sum by 2.
--- step 1: model responded  (2 tool call(s))
  ... THINKING      :    <no summary text; encrypted signature only>
  <<< PREAMBLE      : I’ll look up both cities’ elevations, then add them and divide the sum by 2.
  --> TOOL CALL     : city_info({"city":"Cairo"})   id=resp_muwqj2adjsjvllwttnmg6yxttyk554ykokof4xdcpolsbozt3bhq:call_d23155722cf352178edd0b4e06705868
  --> TOOL CALL     : city_info({"city":"Baltimore"})   id=resp_muwqj2adjsjvllwttnmg6yxttyk554ykokof4xdcpolsbozt3bhq:call_90cf18e03e23597aa638e69057d08796
      usage this call : in 205 (cache read 0, cache write 0) | out 100 (reasoning 21) $0.00018
      usage so far    : requests=1 tool_calls=0 in=205 out=100 cache_read=0

--- step 2: sending to model (2 part(s))
  <-- TOOL RESULT   : city_info